# PRC Data Challenge 2025: reproduce submission v6 and its corrected validation

Run in Google Colab (*Runtime → Run all*). The notebook:

1. downloads the challenge data from Zenodo (record 19184662, CC BY 4.0, about 4 GB for the train and ranking trajectories);
2. builds the segment features (cached in your Drive, since the training set takes a while);
3. trains the v6 model with the **original** procedure and reports its validation RMSE;
4. predicts the ranking intervals and scores them against the published ground truth (`fuel_rank.parquet`);
5. runs the **corrected** flight-grouped cross-validation;
6. optionally rescores your past submission files.

The ANP aircraft files (`Aircraft.csv`, `Aerodynamic_coefficients.csv`) are not redistributed: download them from the
[EASA ANP site](https://www.easa.europa.eu/en/domains/environment/policy-support-and-research/aircraft-noise-and-performance-anp-data)
and put them in `ANP_DIR` below.

**Do not put any access key in this notebook.** The challenge bucket credentials are not needed here.

In [ ]:
!pip -q install --force-reinstall --no-deps git+https://github.com/MoustaphaRouf/prc2025-fuel-burn

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/MES_RECHERCHES/PRC_2025')   # your challenge folder: searched for files you already have
FEATURES = ROOT / 'DOSSIER_FEATURES'                            # train2.parquet / test1.parquet saved by the original notebook
ANP_DIR = ROOT                                                  # Aircraft.csv, Aerodynamic_coefficients.csv
SUBMISSIONS = ROOT                                              # honorable-yogurt_v*.parquet (optional)
WORK = ROOT / 'repro'                                           # outputs of this notebook
DATA = Path('/content/data')                                    # only for files not found under ROOT
WORK.mkdir(parents=True, exist_ok=True)
print(ROOT.exists(), sorted(p.name for p in ROOT.iterdir())[:30])

In [ ]:
from prcfuel import data
# Small tables: taken from ROOT when present, otherwise downloaded (a few MB).
# fuel_rank.parquet (ranking intervals with ground truth) was published after the challenge.
files = data.local_or_download(['fuel_train', 'fuel_rank', 'flightlist_train', 'flightlist_rank'], ROOT, DATA)
fuel_train = data.read_fuel(files['fuel_train'])
fuel_rank = data.read_fuel(files['fuel_rank'])
fl_train = data.read_flight_list(files['flightlist_train'])
fl_rank = data.read_flight_list(files['flightlist_rank'])
print(len(fuel_train), 'training intervals,', len(fuel_rank), 'ranking intervals')

In [ ]:
import pandas as pd
from prcfuel import features

def load_or_build(saved, cache, fuel, flist, traj_dir, zip_key):
    # 1. features saved by the original v6 notebook: no trajectory needed (fastest, and exactly what v6 used)
    if saved.exists():
        print('using', saved)
        return data.add_flight_id(pd.read_parquet(saved), fuel)
    # 2. features already rebuilt by this notebook
    if cache.exists():
        return pd.read_parquet(cache)
    # 3. rebuild from per-flight trajectory files in your Drive, or from the Zenodo archive
    paths = sorted(Path(traj_dir).glob('*.parquet')) if traj_dir else []
    if not paths:
        z = data.download([zip_key], DATA)[zip_key]
        paths = data.extract(z, DATA / zip_key)
    print(len(paths), 'trajectory files')
    df = features.build_dataset(paths, fuel, flist)
    df.to_parquet(cache)
    return df

train_feats = load_or_build(FEATURES / 'train2.parquet', WORK / 'train_features.parquet',
                            fuel_train, fl_train, ROOT / 'Data', 'flights_train')
rank_feats = load_or_build(FEATURES / 'test1.parquet', WORK / 'rank_features.parquet',
                           fuel_rank.drop(columns='fuel_kg'), fl_rank, ROOT / 'FLIGHT_RANK', 'flights_rank')
print(train_feats.shape, rank_feats.shape, 'flight_id missing:', train_feats.flight_id.isna().sum())
print(len(fuel_rank) - rank_feats.idx.isin(fuel_rank.idx).sum(), 'ranking intervals without features (fallback)')

In [ ]:
from prcfuel import aircraft
perfo = aircraft.performance_table(aircraft.load_anp(ANP_DIR / 'Aircraft.csv',
                                                     ANP_DIR / 'Aerodynamic_coefficients.csv'))
perfo.head()

## 1. Original v6 procedure

Random row-level 80/20 split, early stopping on the same 20 %, ranking features clipped with their own quantiles.
The notebook reported a validation RMSE of 177.2 kg; the official score of the v6 file was 263.8 kg.

In [ ]:
from prcfuel import model, score
m, val_rmse, pred = model.reproduce_v6(train_feats, rank_feats, perfo)
fb = model.fallback_predictions(fuel_rank.drop(columns='fuel_kg'), fuel_train, fl_rank, fl_train)
sub = model.submission(fuel_rank[['idx', 'flight_id', 'start', 'end']], pred, fb)
s = score.score(sub, fuel_rank)
print(f'validation RMSE (row split): {val_rmse:.1f} kg')
print(f'ranking RMSE: {s["rmse_kg"]:.1f} kg on {s["matched"]} intervals ({sub.attrs["n_fallback"]} from the fallback)')
sub.to_parquet(WORK / 'v6_reproduced.parquet')

## 2. Corrected validation

Folds grouped by flight, clipping and target encoding fitted on the training fold only, early stopping on an inner
split of the training fold. This estimate should be much closer to the ranking RMSE than the row-split one.

In [ ]:
cv = model.grouped_cv(train_feats, perfo, n_splits=5)
cv.to_csv(WORK / 'grouped_cv.csv', index=False)
print(cv.round(1).to_string(index=False))
print(f'grouped CV RMSE, intervals < 10,000 kg: {cv.rmse_y_lt_10000.mean():.1f} ± {cv.rmse_y_lt_10000.std():.1f} kg')
print(f'all intervals (dominated by a few aberrant training labels): {cv.rmse.mean():.0f} kg')

## 3. Rescore past submissions (optional)

In [ ]:
rows = [{'file': p.name, **score.score(pd.read_parquet(p), fuel_rank)}
        for p in sorted(SUBMISSIONS.glob('honorable-yogurt_v*.parquet'))]
pd.DataFrame(rows).round(1)

Share the printed numbers (not this executed notebook) to update the README.